In [1]:
import pandas as pd
import numpy as np
import warnings
import random
warnings.filterwarnings("ignore")

In [3]:
nacc = pd.read_csv('../../data/new_nacc_unique_type_3.csv')
nacc_test = pd.read_csv('/projectnb/vkolagrp/projects/adrd/NMED_submission_github/data/train_vld_test_split_updated/nacc_test_with_np_cli.csv')

In [4]:
len(nacc)

47165

In [2]:
import json

data = []
with open("../../data/nacc_unique_with_llama_summaries_3/nacc_unique_with_llama_summaries_3.json", "r") as file:
    for line in file:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

# Now `data` is a list of dictionaries, each dictionary is one JSON object from the file

In [3]:
print(f'Length of data before cleaning: {len(data)}')

Length of data before cleaning: 57344


In [4]:
# Load the summaries from version 2
data_ver_2 = []
with open("../../data/nacc_unique_with_llama_summaries_2/nacc_unique_with_llama_summaries_2.json", "r") as file:
    for line in file:
        try:
            data_ver_2.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

In [5]:
print(f'Length of data from version 2: {len(data_ver_2)}')

Length of data from version 2: 47165


## Identify the indices of the responses with repeated sentences

In [9]:
responses = []
for i in range(len(data)):
    responses.append(data[i]['diag_LLAMA_SUMMARY'])

In [10]:
import nltk
from collections import defaultdict

# Ensure you have the Punkt tokenizer models downloaded
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     /usr3/graduate/skowshik/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
nums = ['1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.', '9.']
def find_highly_repeated_sentences(responses):
    highly_repeated = defaultdict(list)
    
    for i, response in enumerate(responses):
        # Tokenize the response into sentences
        sentences = nltk.sent_tokenize(response)
        sentence_counts = defaultdict(int)
        
        # Count each sentence
        for sentence in sentences:
            if sentence in nums:
                continue
            sentence_counts[sentence] += 1
        
        # Check for sentences with more than two repetitions
        for sentence, count in sentence_counts.items():
            if count > 4:
                highly_repeated[i].append((sentence, count))

    return highly_repeated

repeat_indices = []

# Find sentences repeated more than 4 times
repeated_sentences = find_highly_repeated_sentences(responses)

# Print the results
for index, repeats in repeated_sentences.items():
    repeat_indices.append(index)
    # print(f"Response {index}:")
    # for sentence, count in repeats:
    #     print(f"  Repeated Sentence: '{sentence}' (Repeated {count} times)")

In [12]:
len(repeat_indices)

13

In [13]:
for i in repeat_indices:
    if i % 2 != 0:
        print(i)

6907


## Clean the responses

In [14]:
indices_to_delete = []
for i, ind in enumerate(repeat_indices):
    if (data_ver_2[ind // 2]['NACCID'] == data[ind]['NACCID']):
        # Replace repeated responses with the previous version if the instruction matches
        if (data[ind]['instruction'] == 'Provide a cognitive diagnosis for a patient presenting with the following information.'):
            data[ind]['diag_LLAMA_SUMMARY'] = data_ver_2[ind // 2]['diag_LLAMA_SUMMARY']
        # Delete the repeated responses if the instruction does not match
        else:
            print(i, ind, data[ind]['instruction'])
            indices_to_delete.append(ind)

2 6907 Does this patient have multiple systems atrophy?


In [15]:
# Delete the responses by ordering them in descending order
def delete_by_indices(lst, indices):
    # Sort the indices in descending order to avoid index shift issues
    for index in sorted(indices, reverse=True):
        if index < len(lst):
            del lst[index]
            
delete_by_indices(data, indices_to_delete)

In [16]:
# Replace 30% of new responses with the old ones to introduce variety
cnt = 0
modified_indices = []
for ind, entry in enumerate(data):
    if (data_ver_2[ind // 2]['NACCID'] == data[ind]['NACCID']):
        if (data[ind]['instruction'] == 'Provide a cognitive diagnosis for a patient presenting with the following information.'):
            if random.random() < 0.3:
                modified_indices.append(ind)
                cnt += 1
                data[ind]['diag_LLAMA_SUMMARY'] = data_ver_2[ind // 2]['diag_LLAMA_SUMMARY']

In [18]:
print(f'Number of responses replaced with version 2: {len(modified_indices)}')

Number of responses replaced with version 2: 1034


In [17]:
print(f'Length of data After cleaning: {len(data)}')

Length of data After cleaning: 57343


### Check any response

In [45]:
index = 20
nacc[(nacc['NACCID'] == data[index]['NACCID']) & (nacc['NACCUDSD'] == data[index]['NACCUDSD'])][['NACCID', 'NACCUDSD', 'NACCALZP', 'DEPIF', 'FTDIF', 'CDRGLOB', 'PARK', 'VASCIF', 'STROKIF', 'VASCPSIF', 'CVDIF', 'NACCLBDP', 'FTLDNOIF', 'PPAPHIF', 'PSPIF', 'CORTIF', 'NACCMMSE', 'NACCAGE', 'EDUC', 'NACCNE4S']]

,NACCID,NACCUDSD,NACCALZP,DEPIF,FTDIF,CDRGLOB,PARK,VASCIF,STROKIF,VASCPSIF,CVDIF,NACCLBDP,FTLDNOIF,PPAPHIF,PSPIF,CORTIF,NACCMMSE,NACCAGE,EDUC,NACCNE4S
10,NACC000236,2,7,1,7,0.5,0,7,7,7,-4,7,-4,7,7,7,15,65,8,0


In [44]:
print(data[20]['diag_LLAMA_SUMMARY'])

Based on the provided information, I will provide a cognitive diagnosis for the patient.

**Cognitive Diagnosis:**

The patient is a 65-year-old female with a primary language of Spanish, who presents with a gradual onset of cognitive decline. The patient's cognitive assessment scores indicate significant impairment in multiple cognitive domains, including memory, attention, executive function, and language. The patient's MMSE score is 15/30, indicating moderate cognitive impairment. The patient's MoCA score is not reported, but the patient's cognitive assessment scores suggest significant impairment in multiple cognitive domains.

**Cognitive Domains Affected:**

* Memory: The patient's Logical Memory IIA - Delayed score is 1/5, indicating significant impairment in memory.
* Attention: The patient's Digit Span Forward score is 4/9, indicating mild impairment in attention.
* Executive Function: The patient's Trail Making Test Part A score is 113 seconds, indicating significant impairme

## Save train and test datasets

In [ ]:
data_train = []
data_test = []
for entry in data:
    if entry['NACCID'] in (list(nacc_test['ID'])):
        data_test.append(entry)
    else:
        data_train.append(entry)

In [ ]:
json_name_train = "./../data/nacc_unique_with_llama_summaries_3/nacc_unique_with_llama_summaries_3_train.json"
with open(json_name_train, 'w', encoding='utf-8') as json_file:
    json.dump(data_train, json_file)

json_name_test = "./../data/nacc_unique_with_llama_summaries_3/nacc_unique_with_llama_summaries_3_test.json"
with open(json_name_test, 'w', encoding='utf-8') as json_file:
    json.dump(data_test, json_file)

In [ ]:
# index = 10, 0 # LBD DE
# index = 3 # MCI
# index = 12 # AD DE
# index = 1 # NC
index = 242 # NC

nacc[(nacc['NACCID'] == data_train[index]['NACCID']) & (nacc['NACCUDSD'] == data_train[index]['NACCUDSD'])][['NACCID', 'NACCUDSD', 'NACCALZP', 'DEPIF', 'FTDIF', 'CDRGLOB', 'PARK', 'VASCIF', 'STROKIF', 'VASCPSIF', 'CVDIF', 'NACCLBDP', 'FTLDNOIF', 'PPAPHIF', 'PSPIF', 'CORTIF', 'NACCMMSE', 'NACCAGE', 'EDUC', 'NACCNE4S']]

,NACCID,NACCUDSD,NACCALZP,DEPIF,FTDIF,CDRGLOB,PARK,VASCIF,STROKIF,VASCPSIF,CVDIF,NACCLBDP,FTLDNOIF,PPAPHIF,PSPIF,CORTIF,NACCMMSE,NACCAGE,EDUC,NACCNE4S
40372,NACC853155,4,7,7,7,0.5,0,7,7,1,-4,7,-4,7,7,7,22,74,16,0
